# Assistant de recherche cosmétique en pharmacie

Ce notebook fournit un **prototype local** qui recherche des produits dans le catalogue Open Beauty Facts préparé dans `data_Open Beauty Facts/products_clean.csv`. Il n’utilise ni API externe, ni modèle de langage, ni connexion réseau.

**Limites importantes :** les résultats sont des candidats à examiner par un pharmacien, pas des recommandations médicales. Le catalogue peut être incomplet, bruité ou obsolète. Une composition absente ou un marqueur absent ne prouve jamais l’absence d’un ingrédient, une absence d’allergène ou la sécurité d’un produit. Vérifier l’emballage actuel et les sources professionnelles avant tout conseil.

**Utilisation :** exécuter les cellules dans l’ordre, puis lancer `chat()` dans la dernière cellule et saisir `quitter` pour terminer. Les exemples ci-dessous sont fictifs.

Source des données : [Open Beauty Facts](https://world.openbeautyfacts.org/data), base sous licence ODbL. Respecter l’attribution et les conditions de réutilisation de la base.

In [1]:
from pathlib import Path
import re
import unicodedata

import pandas as pd
from IPython.display import display

pd.set_option('display.max_colwidth', 240)
pd.set_option('display.max_columns', 20)
print('pandas', pd.__version__)

pandas 3.1.1


## Charger le catalogue local

Le chemin est recherché depuis le dossier courant et ses parents : le notebook peut donc être ouvert depuis la racine du dépôt ou depuis un sous-dossier.

In [2]:
DATA_RELATIVE_PATH = Path('data_Open Beauty Facts') / 'products_clean.csv'
DATA_PATH = next(
    (base / DATA_RELATIVE_PATH for base in (Path.cwd(), *Path.cwd().parents)
     if (base / DATA_RELATIVE_PATH).is_file()),
    None,
)

if DATA_PATH is None:
    raise FileNotFoundError(
        'Catalogue introuvable. Placez le notebook dans le dépôt contenant '
        "'data_Open Beauty Facts/products_clean.csv'."
    )

catalogue = pd.read_csv(DATA_PATH, dtype=str, encoding='utf-8', encoding_errors='replace')
required_columns = {'product_name', 'ingredients_text'}
missing_columns = required_columns.difference(catalogue.columns)
if missing_columns:
    raise ValueError(f'Colonnes obligatoires absentes du catalogue : {sorted(missing_columns)}')

for column in catalogue.columns:
    catalogue[column] = catalogue[column].fillna('').astype(str)

print(f'Catalogue chargé : {len(catalogue):,} produits')
print(f'Fichier : {DATA_PATH}')
print('Colonnes :', ', '.join(catalogue.columns))
display(catalogue[['product_name', 'brands', 'categories_tags', 'ingredients_text']].head(5))

Catalogue chargé : 18,891 produits
Fichier : c:\Users\mamad\Desktop\DIT\Projet_de_fin_Master 1 IA Jan26\data_Open Beauty Facts\products_clean.csv
Colonnes : code, url, last_modified_t, product_name, quantity, categories_tags, countries_tags, ingredients_text, brands, n_ingredients, ingredients_list_str, allergens_str, sensitive_flags_str


,product_name,brands,categories_tags,ingredients_text
0,Süßlupinen Mehl,bodylab-24,,"CARATTERISTICHE PRODOTTO: gel a base di acqua di Arnica montana (98%) potenziato con glio io brasiliano molto ricco di caroteni) essenziale di timo, mentolo e con olio di buriti (un water-based gel (98%), enhanced with thym PRODUCT CHAR..."
1,pure collagène marin,,,"Nutritional Information Magnesium Stearate and Silicon Dioxide, Ingredients: Marine Collagen Hydrolysate *Nutrient Reference Value † NRV Not Established serving Size: 2 Capsules servings Per Container: 60 Amount Per Serving %NRV Marine ..."
2,Base lavante,aroma-zone,"en:hygiene,en:soaps,fr:savons-liquides","aqua, lauryl glucoside, rosa damascena flower water, tilia vulgaris water, glycerin, sodium cocoamphoacetate, decyl glucoside, lippia citriodora water, sodium cocoyl glutamate, sodium lauryl glucose carboxylate, glyceryl caprylate, leuc..."
3,Colgate toothpaste,,,"ns. a inactive ingredients water, sorbitol, hydrated silica, glycerin, peg-12, tetrasodium pyrophosphate, flavor, sodium lauryl sulfate, zinc phosphate, cellulose gum, sodium citrate, microcrystalline cellulose, sodium saccharin, cocami..."
4,"Nutricentials, day dream protective cream",nu-skin,"en:non-food-products,en:open-beauty-facts","Réservé à un usage exter Ingredients: Aqua, Homosalate, Ethylhexyl Salicylate, Polyglyceryl-6 Stearate, Buy Methoxydibenzoylmethane, Butylene Glycol, Octocrylene, Butyloctyl Salicylate, C12-15 Alkyl Benzoate, Cetyl Dimethicone, Isodecyl..."


## Recherche explicable

Le score lexical privilégie les correspondances dans le nom et la marque, puis dans la catégorie et la composition. Les champs sont indexés une seule fois pour accélérer les échanges. La recherche ne comprend pas les synonymes ni la morphologie : essayez plusieurs mots-clés. Le filtre de pays et l’exclusion d’un ingrédient servent uniquement à réduire la liste de candidats ; ils ne constituent pas une vérification d’allergie ou de sécurité. Si la composition est absente, le produit est exclu d’une recherche avec ingrédient à éviter, mais une composition présente peut rester incomplète.

In [3]:
STOP_WORDS = {
    'avec', 'dans', 'pour', 'sans', 'une', 'des', 'les', 'sur', 'qui', 'que',
    'mon', 'ma', 'mes', 'du', 'de', 'la', 'le', 'un', 'et', 'est', 'au', 'aux',
    'je', 'veux', 'cherche', 'recherche', 'produit', 'peau', 'visage',
}

def normalize_text(value):
    """Lowercase and remove accents for approximate keyword matching."""
    text = unicodedata.normalize('NFKD', str(value)).encode('ascii', 'ignore').decode('ascii')
    return text.lower()


def tokens(value):
    return [word for word in re.findall(r'[a-z0-9]+', normalize_text(value))
            if len(word) > 1 and word not in STOP_WORDS]


SEARCH_WEIGHTS = {
    'product_name': 4,
    'brands': 3,
    'categories_tags': 3,
    'ingredients_text': 1,
    'ingredients_list_str': 1,
}


def _prepare_search_index(products):
    """Normalize searchable fields once so each chat query stays responsive."""
    indexed = products.copy().reset_index(drop=True)
    for field in SEARCH_WEIGHTS:
        values = indexed[field] if field in indexed.columns else pd.Series('', index=indexed.index)
        indexed[f'_search_{field}'] = values.fillna('').astype(str).map(normalize_text)
    country_values = indexed['countries_tags'] if 'countries_tags' in indexed.columns else pd.Series('', index=indexed.index)
    indexed['_search_country'] = country_values.fillna('').astype(str).map(normalize_text)
    indexed['_search_ingredients'] = (
        indexed['_search_ingredients_text'] + ' ' + indexed['_search_ingredients_list_str']
    ).str.strip()
    return indexed


catalogue_index = _prepare_search_index(catalogue)


def search_products(query, products=None, country=None, exclude_ingredients=None, top_k=5):
    """Return ranked catalogue matches; an empty/unknown query never generates products."""
    query_tokens = list(dict.fromkeys(tokens(query)))
    if not query_tokens:
        return pd.DataFrame()
    indexed = catalogue_index if products is None else _prepare_search_index(products)
    if indexed.empty:
        return pd.DataFrame()

    score = pd.Series(0, index=indexed.index, dtype='int64')
    for token in query_tokens:
        pattern = rf'(?<![a-z0-9]){re.escape(token)}(?![a-z0-9])'
        for field, weight in SEARCH_WEIGHTS.items():
            matched = indexed[f'_search_{field}'].str.contains(pattern, regex=True, na=False)
            score += matched.astype('int64') * weight

    candidates = indexed.loc[score > 0].copy()
    candidates['match_score'] = score.loc[score > 0]

    if country:
        country_term = normalize_text(country).strip()
        candidates = candidates.loc[
            candidates['_search_country'].str.contains(country_term, regex=False, na=False)
        ]

    exclusions = [normalize_text(item).strip() for item in (exclude_ingredients or []) if str(item).strip()]
    if exclusions:
        has_composition = candidates['_search_ingredients'].ne('')
        includes_excluded = pd.Series(False, index=candidates.index)
        for item in exclusions:
            includes_excluded |= candidates['_search_ingredients'].str.contains(item, regex=False, na=False)
        candidates = candidates.loc[has_composition & ~includes_excluded]

    if candidates.empty:
        return pd.DataFrame()
    return (candidates
            .sort_values(['match_score', 'product_name'], ascending=[False, True])
            .head(max(1, int(top_k)))
            .drop(columns=[column for column in candidates.columns if column.startswith('_search_')])
            .reset_index(drop=True))


def show_candidates(results):
    """Display candidates with provenance and explicit data-quality cautions."""
    if results.empty:
        print('Aucun résultat dans le catalogue pour ces mots-clés. Essayez une autre formulation ou vérifiez manuellement.')
        return

    for number, (_, product) in enumerate(results.iterrows(), start=1):
        name = product.get('product_name', '') or 'Nom non renseigné'
        brand = product.get('brands', '') or 'Marque non renseignée'
        print(f'\n{number}. {name} — {brand} (score lexical : {product["match_score"]})')
        print('Catégories :', product.get('categories_tags', '') or 'non renseignées')
        print('Composition fournie :', product.get('ingredients_text', '') or 'NON RENSEIGNÉE — impossible de conclure sur les ingrédients')
        print('Source produit :', product.get('url', '') or 'URL non renseignée')
        print('Marqueurs indicatifs dans le jeu de données :')
        print('  allergènes :', product.get('allergens_str', '') or 'aucun marqueur listé (cela ne prouve pas leur absence)')
        print('  marqueurs sensibles :', product.get('sensitive_flags_str', '') or 'aucun marqueur listé (liste indicative et incomplète)')
    print('\nÀ vérifier par le pharmacien sur l’emballage actuel. Ces résultats ne constituent pas un avis médical ni une garantie de sécurité.')


def recommend(query, country=None, exclude_ingredients=None, top_k=5):
    """Search and display candidate products for pharmacist review."""
    results = search_products(query, country=country,
                              exclude_ingredients=exclude_ingredients, top_k=top_k)
    show_candidates(results)
    return results

## Exemples de recherche (scénarios fictifs)

Ces exemples testent seulement la recherche textuelle du catalogue. Ils ne valident pas l’adéquation d’un produit pour une personne.

In [4]:
# Exécuter un exemple à la fois et essayer d'autres mots-clés selon le catalogue.
recommend('crème hydratante visage', top_k=3)
# recommend('nettoyant peau sensible', country='france', top_k=3)


1. crème hydratante visage — zorah (score lexical : 11)
Catégories : en:open-beauty-facts,fr:creme
Composition fournie : eau traité, cire émulsifiante, huile de jojiba, provitamine B5, glycérine, propylène glycol, oxyde de zinc, parfum, conservateur
Source produit : http://world-en.openbeautyfacts.org/product/6132262400046/creme-hydratante-visage-zorah
Marqueurs indicatifs dans le jeu de données :
  allergènes : parfum
  marqueurs sensibles : aucun marqueur listé (liste indicative et incomplète)

2. Crème régénératrice peau des mains — ONLYBIO (score lexical : 10)
Catégories : fr:crème hydratante pour mains
Composition fournie : Aqua, Helianthus Annuus (Sunflower) Seed Oil, Cetearyl Alcohol, Butyrospermum Parkii (Shea) Butter, Glycerin, Solanum Tuberosum Starch, Sodium Cetearyl Sulfate, Propanediol, Octyldodecanol, Cannabidiol, Glyceryl Stearate, Levan, Tocopherol, Glucose, Cannabis Sativa Seed Oil, Sucrose, Benzyl Acohol, Dehydroacetic Acid, Citric Acid, Fructose, Benzoic Acid, Parfu

,code,url,last_modified_t,product_name,quantity,categories_tags,countries_tags,ingredients_text,brands,n_ingredients,ingredients_list_str,allergens_str,sensitive_flags_str,match_score
0,6132262400046,http://world-en.openbeautyfacts.org/product/6132262400046/creme-hydratante-visage-zorah,2025-02-19T19:25:59Z,crème hydratante visage,65 ml,"en:open-beauty-facts,fr:creme",en:algeria,"eau traité, cire émulsifiante, huile de jojiba, provitamine B5, glycérine, propylène glycol, oxyde de zinc, parfum, conservateur",zorah,9,eau traite; cire emulsifiante; huile de jojiba; provitamine b5; glycerine; propylene glycol; oxyde de zinc; parfum; conservateur,parfum,,11
1,5902811788373,http://world-en.openbeautyfacts.org/product/5902811788373/creme-regeneratrice-peau-des-mains-onlybio,2026-07-18T16:46:34Z,Crème régénératrice peau des mains,75 ml,fr:crème hydratante pour mains,en:france,"Aqua, Helianthus Annuus (Sunflower) Seed Oil, Cetearyl Alcohol, Butyrospermum Parkii (Shea) Butter, Glycerin, Solanum Tuberosum Starch, Sodium Cetearyl Sulfate, Propanediol, Octyldodecanol, Cannabidiol, Glyceryl Stearate, Levan, Tocophe...",ONLYBIO,25,aqua; helianthus annuus seed oil; cetearyl alcohol; butyrospermum parkii butter; glycerin; solanum tuberosum starch; sodium cetearyl sulfate; propanediol; octyldodecanol; cannabidiol; glyceryl stearate; levan; tocopherol; glucose; canna...,citral; geraniol; limonene; parfum,aha_bha_peel,10
2,3665467000733,http://world-en.openbeautyfacts.org/product/3665467000733/creme-hydratante-visage-typology,2019-05-21T08:31:54Z,crème hydratante visage,50 ml,,en:france,"de coco bio, prévient la A(COCO déshydratation de la peau cOConut oil, prevents skin from dehydration. 3. GLYCERIN: Issue du tournesol, permet i superficielle de la peau./Derive sunflower, hydrates the superis of the skin. 4. CETYL ALCO...",typology,8,de coco bio; previent la a(coco deshydratation de la peau coconut oil; prevents skin from dehydration. 3. glycerin: issue du tournesol; permet i superficielle de la peau./derive sunflower; hydrates the superis of the skin. 4. cetyl alco...,,,10


## Lancer le chatbot

Exécutez la cellule suivante puis saisissez une demande en français. Commandes disponibles : `aide`, `quitter`. Une demande peut être une recherche par mots-clés, par exemple `crème hydratante visage`. Le chatbot fait une recherche dans le catalogue ; il ne pose pas de diagnostic.

In [ ]:
def chat():
    """Start a simple interactive, local keyword-search chat session."""
    print('Assistant de recherche cosmétique — aide à la recherche, pas un avis médical.')
    print("Saisissez 'aide' pour les commandes ou 'quitter' pour terminer.\n")
    while True:
        try:
            message = input('Vous : ').strip()
        except (EOFError, KeyboardInterrupt):
            print('\nFin de la session.')
            break
        if normalize_text(message) in {'quitter', 'exit', 'stop'}:
            print('Fin de la session. Vérifiez les informations avec le pharmacien.')
            break
        if normalize_text(message) in {'aide', 'help'}:
            print('Décrivez le type de produit ou les ingrédients à rechercher. Exemples fictifs : crème hydratante visage ; nettoyant doux. Essayez plusieurs mots-clés si aucun résultat ne correspond.')
            continue
        if not tokens(message):
            print('Précisez quelques mots-clés pour rechercher dans le catalogue.')
            continue
        recommend(message, top_k=5)
        print('\nVous pouvez préciser votre recherche ou saisir une nouvelle demande.\n')

# Lancer interactivement en exécutant :
# chat()

# Vérifications rapides

Les tests utilisent un petit catalogue fictif pour vérifier le classement, la requête vide et la gestion de colonnes manquantes. Ils ne constituent pas une validation médicale.

In [6]:
test_catalogue = pd.DataFrame([
    {'product_name': 'Crème hydratante visage', 'brands': 'Marque A',
     'categories_tags': 'soin visage', 'ingredients_text': 'aqua, glycerin',
     'url': 'https://example.invalid/a'},
    {'product_name': 'Shampooing', 'brands': 'Marque B',
     'categories_tags': 'cheveux', 'ingredients_text': 'aqua',
     'url': 'https://example.invalid/b'},
])

ranked = search_products('crème hydratante', products=test_catalogue)
assert not ranked.empty and ranked.iloc[0]['product_name'] == 'Crème hydratante visage'
assert search_products('   ', products=test_catalogue).empty
assert search_products('inconnu', products=test_catalogue).empty
assert search_products('hydratante', products=pd.DataFrame([{'product_name': 'Crème hydratante'}])).iloc[0]['product_name'] == 'Crème hydratante'
unknown_composition = pd.DataFrame([{'product_name': 'Crème hydratante', 'ingredients_text': ''}])
assert search_products('hydratante', products=unknown_composition, exclude_ingredients=['parfum']).empty
print('Tests réussis : classement lexical, requête vide, absence de résultat, champs facultatifs et composition inconnue.')

Tests réussis : classement lexical, requête vide, absence de résultat, champs facultatifs et composition inconnue.


## Limites et prochaines étapes

- Le découpage et les marqueurs présents dans le jeu préparé sont indicatifs ; vérifier leur exactitude avant de les utiliser.
- La recherche ne comprend pas les synonymes, les fautes de frappe ni les besoins dermatologiques. Un résultat bien classé n’est pas une preuve d’adéquation.
- Un filtre d’ingrédient ne peut pas confirmer qu’un produit est exempt d’un ingrédient si la composition source est incomplète ou mal extraite.
- Avant tout usage réel : faire valider le catalogue et les règles par des pharmaciens, auditer les données, tester les abstentions, et examiner les obligations applicables en matière de santé, de confidentialité et de réglementation.

Le fichier brut `obf_raw.csv` n’est pas requis pour ce notebook : seul le catalogue nettoyé est utilisé.